# Практическая работа №1
## Логистическая регрессия: проверка банкнот

**Цель работы** — применение логистической регрессии для бинарной классификации банкнот по числовым характеристикам изображения.

В работе был использован набор данных Banknote Authentication. Были проведены проверка качества данных, анализ признаков и распределения классов. Затем данные были разделены на обучающую и тестовую выборки, выполнено обучение модели и получены предсказания. Для оценки результата использовались Accuracy, Precision, Recall, F1-score и ROC-AUC, а также графики, показывающие качество разделения классов.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scipy scikit-learn

Для работы были выбраны библиотеки `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy` и `scikit-learn`. Команда `%pip install` устанавливает их в среду, в которой открыт ноутбук. Параметр `-q` уменьшает количество служебных сообщений при установке.

После запуска ячейки могут появиться сообщения об установке пакетов. Числовых результатов анализа на данном этапе ещё нет: библиотеки только подготавливаются к дальнейшей работе.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)

# В браузерной среде используется open_url, в локальной — pandas.
try:
    from pyodide.http import open_url
except ImportError:
    open_url = None

from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (precision_recall_curve, average_precision_score, roc_curve)
from sklearn.calibration import calibration_curve


Были подключены инструменты для обработки таблиц, вычислений, визуализации и обучения моделей. Для работы с файлами используется `Path`, таблицами — `pandas`, массивами — `numpy`. Графики строятся средствами `matplotlib` и `seaborn`. Из `scikit-learn` импортированы алгоритмы, функции разделения выборки, преобразования признаков и метрики качества. `scipy.stats` применяется для статистических графиков.

Отдельный блок `try / except ImportError` проверяет возможность загрузки файлов через `pyodide.http.open_url`. Это сделано для совместимости с браузерным Jupyter. В обычном Python переменная `open_url` остаётся равной `None`. Если все импорты выполнены без ошибок, ячейка ничего не печатает, но все указанные функции становятся доступны далее.

In [ ]:
def read_csv_compat(url, local_path=None, **read_csv_kwargs):
    if local_path is not None:
        local_path = Path(local_path)
        if local_path.exists():
            return pd.read_csv(local_path, **read_csv_kwargs)

    if open_url is not None:
        data = pd.read_csv(open_url(url), **read_csv_kwargs)
    else:
        data = pd.read_csv(url, **read_csv_kwargs)

    if local_path is not None:
        local_path.parent.mkdir(parents=True, exist_ok=True)
        data.to_csv(local_path, index=False)

    return data


Была задана функция `read_csv_compat`, которая используется при загрузке всех четырёх наборов данных. Сначала проверяется наличие локального CSV. Если он уже сохранён, таблица читается с диска. При отсутствии файла выполняется загрузка по URL: в браузерной среде через `open_url`, в обычном Jupyter через `pd.read_csv`. После загрузки данные записываются в папку `data`.

Аргумент `**read_csv_kwargs` позволяет при необходимости передавать дополнительные настройки чтения CSV. Команда `return data` возвращает подготовленную таблицу `DataFrame`. На экране после объявления функции ничего не появляется: фактическое чтение будет выполнено при её вызове.

## 1. Загрузка данных

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

data_path = DATA_DIR / "bank_notes.csv"
data_url = "https://raw.githubusercontent.com/AbhiRoy96/Banknote-Authentication-UCI-Dataset/master/bank_notes.csv"

df = read_csv_compat(data_url, data_path)
df.head()


Был указан адрес исходного CSV и путь к локальной копии. Выражение `DATA_DIR.mkdir(exist_ok=True)` создаёт папку `data`, если её ещё нет. Функция `read_csv_compat` возвращает таблицу в переменную `df`. Завершающая команда `df.head()` показывает пять первых строк.

В выводе отображаются названия столбцов и значения первых пяти объектов. По этой таблице проверяется правильность загрузки: не съехали ли столбцы, как представлены признаки и где находится целевая переменная. Это первичный осмотр данных, а не результат работы алгоритма.

## 2. Первичный анализ

In [ ]:
print("Размер датасета:", df.shape)
display(df.head())
df.info()

Для первичной проверки были использованы `df.shape`, `df.head()` и `df.info()`. Первая команда показывает размер набора данных в виде `(число строк, число столбцов)`. Вторая выводит небольшой фрагмент таблицы. Третья показывает тип данных каждого столбца и количество непустых значений (`Non-Null Count`).

Если для какого-либо признака число непустых значений меньше общего числа строк, в нём имеются пропуски. Также по `dtype` можно проверить, что числовые признаки действительно распознаны как числа, а названия категорий — как текст.

In [ ]:
df.describe().T

Была получена описательная статистика с помощью `describe()`. Показатель `count` отражает число заполненных значений; `mean` — среднее; `std` — стандартное отклонение; `min` и `max` — границы наблюдаемого диапазона. Квартили `25%`, `50%` и `75%` характеризуют положение основной части данных, при этом `50%` соответствует медиане. Операция `.T` разворачивает таблицу, чтобы признаки располагались строками.

По полученной таблице удобно сравнивать диапазоны значений. Большая разница между средним и медианой может говорить об асимметрии распределения, а крайние минимумы и максимумы требуют дополнительной проверки.

## 3. Проверка данных

In [ ]:
print("Пропуски:")
display(df.isnull().sum().to_frame("missing"))

print("Количество полных дубликатов:", df.duplicated().sum())
print("\nРаспределение классов:")
display(df["Target"].value_counts().sort_index())

Проверка пропусков была выполнена выражением `df.isnull().sum()`: оно подсчитывает отсутствующие значения для каждого столбца. Отдельно командой `df.duplicated().sum()` определяется число полностью совпадающих строк. Таблица `missing` содержит количество пропусков, а следующая строка вывода — число дубликатов.

Нулевые значения в этих результатах означают, что соответствующая проблема не обнаружена. Ненулевые значения показывают, какие столбцы или строки требуют обработки. Для задач классификации дополнительно выводится количество объектов каждого класса: это позволяет оценить их представленность ещё до обучения.

In [ ]:
feature_columns = ["variance", "skewness", "curtosis", "entropy"]

outliers = {}
for column in feature_columns:
    values = df[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    outliers[column] = ((values < low) | (values > high)).sum()

pd.Series(outliers, name="Количество значений за границами IQR")

Для анализа были выбраны четыре числовых признака банкнот: `variance`, `skewness`, `curtosis` и `entropy`. С помощью цикла по этим столбцам были рассчитаны квартили и границы IQR. Условие `(values < low) | (values > high)` подсчитывает наблюдения за допустимыми для этого правила пределами.

В результатах появляется число потенциальных выбросов отдельно по каждому признаку. Такие наблюдения проверяются дополнительно: применение одного формального порога не доказывает, что конкретная строка ошибочна.

In [ ]:
fig, axes = plt.subplots(1, len(feature_columns), figsize=(13, 4))

for ax, column in zip(axes, feature_columns):
    ax.boxplot(df[column].dropna())
    ax.set_title(column)

plt.suptitle("Диаграммы размаха признаков", y=1.03)
plt.tight_layout()
plt.show()

Для каждого из четырёх признаков банкноты отдельно показаны медиана, межквартильный диапазон, «усы» и потенциальные выбросы.

Проверяем, нет ли явно ошибочных значений. Сам факт выхода точки за «усы» не означает, что строку нужно удалить.

In [ ]:
before = len(df)
df = df.drop_duplicates().copy()

print("Строк до удаления дубликатов:", before)
print("Строк после:", len(df))
print("Оставшиеся пропуски:", int(df.isnull().sum().sum()))

Было выполнено удаление полностью повторяющихся строк и строк с отсутствующими значениями. Метод `drop_duplicates()` убирает дубликаты, `dropna()` — записи с `NaN`, а `copy()` создаёт очищенную таблицу.

Вывод показывает число объектов до обработки и после неё. Это позволяет оценить, изменился ли объём выборки. На следующих графиках и при обучении используются уже очищенные данные.

## 4. Разведочный анализ

In [ ]:
df[feature_columns].hist(figsize=(10, 8), bins=25)
plt.suptitle("Распределения признаков", y=1.02)
plt.tight_layout()
plt.show()

Гистограммы показывают распределение каждого числового признака банкноты.

По ним видно, насколько равномерно распределены значения, есть ли сильный перекос и несколько выраженных групп наблюдений.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Target", color=sns.color_palette()[0])
plt.title("Количество объектов каждого класса")
plt.xlabel("Класс")
plt.ylabel("Количество")
plt.show()

Столбцы показывают количество объектов классов `0` и `1`.

Если один столбец был бы намного выше второго, это означало бы сильный дисбаланс классов. Здесь график используется для проверки, насколько равномерно представлены оба класса.

In [ ]:
corr_data = df[feature_columns + ["Target"]].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_data, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Матрица корреляции")
plt.show()

Матрица показывает линейные связи между четырьмя входными признаками и `Target`.

Можно увидеть, какие характеристики сильнее связаны с классом, и проверить наличие сильно взаимосвязанных признаков.

### KDE: дисперсия по двум классам банкнот

In [ ]:
plt.figure(figsize=(9, 5))
for cls, group in df.groupby("Target"):
    sns.kdeplot(data=group, x="variance", label=f"Класс {cls}", fill=False, color=sns.color_palette()[int(cls)])
plt.xlabel("Дисперсия коэффициентов вейвлет-преобразования")
plt.ylabel("Оценка плотности распределения")
plt.title("KDE: дисперсия по двум классам банкнот")
plt.legend()
plt.tight_layout()
plt.show()

Для сравнения использован график: ядерная оценка плотности (*KDE, kernel density estimation*). `groupby("Target")` делит наблюдения на два класса, а `kdeplot` строит отдельные сглаженные плотности для признака `variance`.

X — значение признака, Y — оценка **плотности**, а не число банкнот. Перекрытие кривых обозначает диапазоны, где один признак не позволяет надёжно различать классы.

Чем лучше разделены кривые, тем потенциально информативнее признак. Это не оценка точности всей модели — она учитывает сразу четыре характеристики.

### Диаграмма рассеяния: variance vs skewness, цвет — класс

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="variance", y="skewness", hue="Target", alpha=0.55, s=22)
plt.xlabel("Дисперсия вейвлет-коэффициентов")
plt.ylabel("Асимметрия вейвлет-коэффициентов")
plt.title("Диаграмма рассеяния: variance vs skewness, цвет — класс")
plt.tight_layout()
plt.show()

Была построена диаграмма рассеяния по двум признакам (*scatter plot with class coloring*). `hue="Target"` задаёт цвет по истинному классу, `alpha` уменьшает перекрытие точек.

Каждый маркер — одна банкнота; X — дисперсия, Y — асимметрия, цвет — реальная метка. Смешение цветов означает перекрытие классов в этом двумерном срезе.

По двум признакам может быть сложно разделить классы, но логистическая модель использует все четыре признака одновременно.

### Violin plot: распределение skewness по классам

In [ ]:
plt.figure(figsize=(7, 5))
sns.violinplot(data=df, x="Target", y="skewness", inner="quartile", cut=0, color=sns.color_palette()[0])
plt.xlabel("Истинный класс банкноты")
plt.ylabel("Асимметрия вейвлет-коэффициентов")
plt.title("Violin plot: распределение skewness по классам")
plt.tight_layout()
plt.show()

Была построена скрипичная диаграмма (*violin plot*). Сглаженные плотности отдельно для классов `0` и `1`, квартильные линии нанесены внутри.

По X — класс, по Y — значение асимметрии; ширина силуэта характеризует относительную плотность значений на высоте Y.

Различающиеся формы и медианы показывают различия распределений, но не гарантируют, что по одному признаку можно правильно классифицировать все банкноты.

### ECDF: энтропия изображения по классам

In [ ]:
plt.figure(figsize=(9, 5))
for cls, group in df.groupby("Target"):
    values = np.sort(group["entropy"].to_numpy())
    share = np.arange(1, len(values) + 1) / len(values)
    plt.step(values, share, where="post", label=f"Класс {cls}")
plt.xlabel("Энтропия изображения (признак entropy)")
plt.ylabel("Доля значений ≤ X")
plt.title("ECDF: энтропия изображения по классам")
plt.legend()
plt.ylim(0, 1.02)
plt.tight_layout()
plt.show()

Была построена эмпирическая функция распределения (*ECDF*). Отсортированные значения признака `entropy` преобразуются в накопленную долю объектов каждого класса.

Для каждой линии по X — энтропия, по Y — доля банкнот соответствующего класса, имеющих значение признака не больше X.

Горизонтальный сдвиг линий означает различие распределений; если они совпадают, один признак `entropy` по этому критерию слабо различает группы.

## 5. Подготовка выборки

In [ ]:
X = df[feature_columns]
y = df["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Для классификации выделены `X` — четыре входные характеристики банкноты и `y` — столбец `Target`, содержащий два класса. Функция `train_test_split` распределяет объекты на 80% обучающих и 20% тестовых. Параметр `stratify=y` сохраняет примерно одинаковое соотношение классов в обеих частях, а `random_state=42` фиксирует разбиение.

В выводе показываются размеры `X_train` и `X_test`; в обеих матрицах должно быть по четыре столбца. Тестовые объекты не используются при настройке модели.

## 6. Обучение

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

Была составлена цепочка `Pipeline` из `StandardScaler` и `LogisticRegression`. На первом этапе числовые признаки стандартизируются; параметры стандартизации определяются только по обучающей части. На втором этапе логистическая регрессия подбирает коэффициенты для разделения классов. `max_iter=1000` задаёт максимальное число итераций оптимизации.

После `fit()` метод `predict()` возвращает метки классов и сохраняет их в `y_pred`, а `predict_proba(X_test)[:, 1]` возвращает вероятность класса 1 в `y_prob`. Ничего не печатается, но оба массива далее используются для расчёта качества и построения ROC- и PR-кривых.

## 7. Метрики

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"ROC-AUC:   {roc_auc:.3f}")
print()
print(classification_report(y_test, y_pred, digits=3))

Качество было рассчитано по Accuracy, Precision, Recall, F1 и ROC-AUC. Accuracy — доля правильных ответов. Precision показывает, какая доля объектов, отнесённых к классу 1, действительно принадлежит этому классу. Recall отражает долю найденных объектов класса 1 среди всех настоящих единиц. F1 объединяет Precision и Recall, а ROC-AUC оценивает разделение классов по предсказанным вероятностям.

Сначала выводятся значения пяти метрик, затем `classification_report`. В отчёте строки 0 и 1 показывают показатели по каждому классу, `support` — число объектов этого класса в тесте, `macro avg` — простое среднее между классами, `weighted avg` — среднее с учётом их численности.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["0", "1"],
    yticklabels=["0", "1"]
)
plt.xlabel("Предсказанный класс")
plt.ylabel("Фактический класс")
plt.title("Матрица ошибок")
plt.show()

Матрица ошибок сравнивает фактические и предсказанные классы.

Главная диагональ — правильные ответы. Ячейки вне диагонали — ошибки:
- фактический `0`, предсказан `1`;
- фактический `1`, предсказан `0`.

Чем больше значений на диагонали и меньше вне неё, тем лучше работает классификатор.

Поскольку это бинарная задача, матрица имеет размер 2×2. Вне диагонали отображаются ложноположительные и ложноотрицательные решения.

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC-кривая")
plt.show()

ROC-кривая отображает работу модели при разных порогах перевода вероятности в класс.

Чем ближе кривая проходит к левому верхнему углу, тем лучше модель разделяет классы. Диагональ соответствовала бы почти случайному угадыванию. Численно это качество дополнительно отражает метрика ROC-AUC.

ROC-кривая получена по вероятностям, а не по окончательным меткам. Поэтому она показывает качество разделения при изменении порога классификации.

### Кривая Precision–Recall для класса 1

In [ ]:
pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
plt.figure(figsize=(8, 5))
plt.plot(pr_recall, pr_precision, label=f"Average precision = {ap:.3f}")
plt.xlabel("Полнота (Recall)")
plt.ylabel("Точность положительных предсказаний (Precision)")
plt.title("Кривая Precision–Recall для класса 1")
plt.ylim(0, 1.02)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Была построена кривая точности-полноты (*Precision–Recall curve*). `precision_recall_curve` перебирает пороги вероятности, `average_precision_score` вычисляет сводный показатель AP.

X — Recall, Y — Precision для **положительного класса 1**. Увеличение полноты часто сопровождается снижением точности.

Чем выше кривая при сопоставимой полноте, тем лучше способность модели находить положительный класс без большого числа ложных срабатываний. AP — не то же самое, что ROC-AUC.

### Распределения предсказанных вероятностей по истинным классам

In [ ]:
prob_frame = pd.DataFrame({"Вероятность класса 1": y_prob, "Истинный класс": y_test.to_numpy()})
plt.figure(figsize=(9, 5))
sns.histplot(data=prob_frame, x="Вероятность класса 1", hue="Истинный класс", palette="tab10",
             bins=20, element="step", stat="density", common_norm=False, alpha=0.3)
plt.axvline(0.5, color="black", ls="--", label="Порог 0,5")
plt.xlabel("Предсказанная вероятность класса 1")
plt.ylabel("Плотность внутри класса")
plt.title("Распределения предсказанных вероятностей по истинным классам")
plt.tight_layout()
plt.show()

Были построены наложенные гистограммы вероятностей (*class-conditional probability distributions*). `predict_proba` уже вычислен выше; `stat="density", common_norm=False` нормирует распределение отдельно для каждого класса, а пунктир показывает порог 0.5.

По X — вероятность класса 1, по Y — плотность; хорошо разделённые группы обычно сосредоточены ближе к противоположным краям оси.

Объекты классов, попадающие по «неправильную» сторону 0.5, чаще оказываются классификационными ошибками. Это диагностическое описание именно тестовых предсказаний.

### Нормированная матрица ошибок (по фактическому классу)

In [ ]:
cm_norm = confusion_matrix(y_test, y_pred, normalize="true")
plt.figure(figsize=(7, 5))
sns.heatmap(cm_norm, annot=True, fmt=".1%", cmap="Blues", vmin=0, vmax=1,
            xticklabels=["0", "1"], yticklabels=["0", "1"])
plt.xlabel("Предсказанный класс")
plt.ylabel("Фактический класс")
plt.title("Нормированная матрица ошибок (по фактическому классу)")
plt.tight_layout()
plt.show()

Для сравнения использован график: нормированная тепловая карта ошибок (*row-normalized confusion matrix*). `normalize="true"` делит строки матрицы ошибок на количество объектов соответствующего реального класса.

Числа — доли/проценты внутри каждого **фактического** класса; каждая строка суммируется примерно до 100%.

Диагональные ячейки показывают долю правильно найденных объектов каждого класса (class recall), а боковые — доли ошибок. В отличие от обычной матрицы, проценты удобнее сравнивать, если размеры классов неодинаковы.

### Диаграмма калибровки вероятностей (reliability diagram)

In [ ]:
observed_freq, mean_pred = calibration_curve(y_test, y_prob, n_bins=8, strategy="quantile")
plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], ls="--", color="gray", label="Идеальная калибровка")
plt.plot(mean_pred, observed_freq, marker="o", label="Логистическая регрессия")
plt.xlabel("Средняя предсказанная вероятность класса 1")
plt.ylabel("Фактическая доля класса 1")
plt.title("Диаграмма калибровки вероятностей (reliability diagram)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Была построена диаграмма надёжности вероятностей (*calibration / reliability diagram*). `calibration_curve` делит прогнозы на 8 групп по квантилям и сравнивает среднюю вероятность с наблюдаемой частотой класса 1.

X — средняя вероятность, заявленная моделью; Y — доля реально положительных объектов. Идеальные калиброванные вероятности лежали бы на диагонали.

Отклонения от диагонали указывают на завышение или занижение вероятностей. Это отдельное качество, **не тождественное Accuracy или ROC-AUC**. Точек немного: небольшая выборка делает оценку калибровки шумной.

### Знаки и величины коэффициентов логистической модели

In [ ]:
coef = model.named_steps["logreg"].coef_.ravel()
order_coef = np.argsort(np.abs(coef))
plt.figure(figsize=(8, 5))
plt.barh(np.array(X.columns)[order_coef], coef[order_coef])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Коэффициент логистической регрессии (log-odds)")
plt.ylabel("Стандартизированный признак")
plt.title("Знаки и величины коэффициентов логистической модели")
plt.tight_layout()
plt.show()

Была построена диаграмма коэффициентов модели (*coefficient bar plot*). `named_steps["logreg"].coef_` получает коэффициенты после стандартизации четырёх признаков; `argsort(abs(...))` сортирует по модулю.

Каждый столбец показывает, как изменение стандартизированного признака на одну единицу влияет на **логарифм шансов** класса 1 при прочих равных.

Положительный коэффициент увеличивает логарифм шансов класса 1, отрицательный — уменьшает. Модуль коэффициента сам по себе **не является причинным эффектом**; при взаимосвязанных признаках интерпретация осторожная.

### Сравнение Precision, Recall и F1 по классам

In [ ]:
class_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
labels = ["0", "1"]
positions = np.arange(len(labels))
width = 0.25
plt.figure(figsize=(8, 5))
for offset, name in enumerate(["precision", "recall", "f1-score"]):
    values = [class_report[c][name] for c in labels]
    plt.bar(positions + (offset - 1) * width, values, width, label=name)
plt.xticks(positions, ["Класс 0", "Класс 1"])
plt.ylim(0, 1.05)
plt.ylabel("Значение метрики [0, 1]")
plt.title("Сравнение Precision, Recall и F1 по классам")
plt.legend()
plt.tight_layout()
plt.show()

Для сравнения использован график: сгруппированная столбчатая диаграмма (*grouped bar chart*). `classification_report(..., output_dict=True)` получает отдельные Precision, Recall, F1 для классов 0 и 1; три столбца рядом относятся к одному классу.

X — класс, Y — значение метрики в диапазоне 0–1. Различия столбцов показывают, какую именно характеристику качества модель обеспечивает лучше для каждого класса.

Такая детализация не позволяет высокой общей Accuracy скрыть слабый Recall одного из классов.

## Вывод

Логистическая регрессия была обучена для бинарной классификации банкнот. Перед обучением выполнены проверка и очистка данных, анализ распределений и корреляций. Итоговое качество оценивается сразу несколькими метриками, так как одной Accuracy недостаточно для полного описания классификатора. Матрица ошибок показывает количество правильных и ошибочных решений для каждого класса, а ROC-AUC характеризует способность модели разделять два класса при разных порогах.